In [7]:
import numpy as np

class Conv3x3:

    def __init__(self, num_filters):
        self.num_filters = num_filters

        self.filters = np.random.randn(num_filters, 3, 3) / 9

    def iterate_regions(self, image):
        # valid padding
        h, w = image.shape

        for i in range(h-2):
            for j in range(w-2):
                im_region = image[i:(i+3), j:(j+3)]
                yield im_region, i, j

    def forward(self, input):
        h, w = input.shape
        output = np.zeros((h-2, w-2, self.num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.sum(im_region * self.filters, axis=(1, 2))

        return output
    
    

In [8]:
import os
from PIL import Image

PATH = "./CN_MNIST/data/data"
img_num = 4500
files = os.listdir(PATH)[:img_num]
data = []
labels = []
kinds = 15

for i, file in enumerate(files):
    img = Image.open(PATH + '/' + file)
    data.append(np.array(img))
    _, _, _, label = file.split("_")
    label, _ = label.split(".")
    labels.append(int(label) - 1)

data = np.array(data)



In [9]:
from sklearn.model_selection import train_test_split

train_images, test_images, train_labels, test_labels = train_test_split(data, labels, test_size=0.33)

In [10]:
class MaxPool2:

    def iterate_regions(self, image):
        h, w, _ = image.shape
        new_h = h // 2
        new_w = w // 2

        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2):(i*2+2), (j*2):(j*2+2)]
                yield im_region, i, j

    def forward(self, input):
        h, w, num_filters = input.shape
        output = np.zeros((h // 2, w // 2, num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.amax(im_region, axis=(0, 1))

        return output

In [11]:
class Softmax:

    def __init__(self, input_len, nodes):
        self.weights = np.random.randn(input_len, nodes) / input_len
        self.biases = np.zeros(nodes)

    def forward(self, input):
        input = input.flatten()

        input_len, nodes = self.weights.shape

        totals = np.dot(input, self.weights) + self.biases
        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)


In [12]:
conv = Conv3x3(8)
pool = MaxPool2()
fc = Softmax(31 * 31 * 8, kinds)

def forward(image, label):
    out = conv.forward((image / 255) - 0.5)
    out = pool.forward(out)
    out = fc.forward(out)

    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0

    return out, loss, acc

loss = 0
num_correct = 0

for i, (img, label) in enumerate(zip(test_images,test_labels)):
    _, l, acc = forward(img, label)
    loss += l
    num_correct += acc

    if i % 100 == 99:
        print('[Step %d] Past 100 steps: Average Loss %.3f | Accuracy: %.2f%%' %
              (i+1, loss / 100, num_correct))
        loss = 0
        num_correct = 0

[Step 100] Past 100 steps: Average Loss 2.708 | Accuracy: 7.00%
[Step 200] Past 100 steps: Average Loss 2.708 | Accuracy: 6.00%
[Step 300] Past 100 steps: Average Loss 2.708 | Accuracy: 7.00%
[Step 400] Past 100 steps: Average Loss 2.708 | Accuracy: 3.00%
[Step 500] Past 100 steps: Average Loss 2.708 | Accuracy: 5.00%
[Step 600] Past 100 steps: Average Loss 2.708 | Accuracy: 4.00%
[Step 700] Past 100 steps: Average Loss 2.708 | Accuracy: 6.00%
[Step 800] Past 100 steps: Average Loss 2.708 | Accuracy: 0.00%
[Step 900] Past 100 steps: Average Loss 2.708 | Accuracy: 2.00%
[Step 1000] Past 100 steps: Average Loss 2.708 | Accuracy: 6.00%
[Step 1100] Past 100 steps: Average Loss 2.708 | Accuracy: 8.00%
[Step 1200] Past 100 steps: Average Loss 2.708 | Accuracy: 5.00%
[Step 1300] Past 100 steps: Average Loss 2.708 | Accuracy: 3.00%
[Step 1400] Past 100 steps: Average Loss 2.708 | Accuracy: 8.00%
